In [1]:
import os
import cv2
import numpy as np
import random
import yaml

# ==========================================================
# BUILD YOLO-FORMAT COMBO DATASET FROM RAW SEGMENTATION DATA
# ==========================================================
# Your three attached datasets are all image+mask pairs (no
# bounding boxes, no train/val split). This script:
#   1. Finds image/mask pairs in each source's actual layout
#   2. Converts each mask's foreground blob(s) into YOLO bboxes
#      via cv2.findContours + boundingRect
#   3. Splits into train/val and writes images + label .txt
#      files into /kaggle/working/chakramodel_yolo_combo_dataset
#   4. Writes a dataset.yaml pointing at that folder
#
# Run this ONCE per session before kaggle_yolo_retrain.py, and
# point that script's dataset_path at the OUTPUT_DIR below.
# ==========================================================

OUTPUT_DIR = "/kaggle/working/chakramodel_yolo_combo_dataset"
VAL_FRACTION = 0.15
MIN_CONTOUR_AREA = 30  # px^2 -- ignores mask speckle noise
SEED = 42

# ---- Each source has its own image/mask layout; declare pairs explicitly ----
SOURCES = [
    {
        "name": "cvc-clinicdb",
        "images_dir": "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Original",
        "masks_dir": "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Ground Truth",
    },
    {
        "name": "cvc-300",
        "images_dir": "/kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset/CVC-300/images",
        "masks_dir": "/kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset/CVC-300/masks",
    },
    {
        "name": "etis-laribpolypdb",
        "images_dir": "/kaggle/input/datasets/nguyenvoquocduong/etis-laribpolypdb/images",
        "masks_dir": "/kaggle/input/datasets/nguyenvoquocduong/etis-laribpolypdb/masks",
    },
]

IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


def find_pairs(images_dir, masks_dir):
    """Match by exact basename (without extension) between two sibling dirs."""
    if not os.path.isdir(images_dir) or not os.path.isdir(masks_dir):
        return []
    img_map = {os.path.splitext(f)[0]: f for f in os.listdir(images_dir) if f.lower().endswith(IMG_EXTS)}
    mask_map = {os.path.splitext(f)[0]: f for f in os.listdir(masks_dir) if f.lower().endswith(IMG_EXTS)}
    pairs = []
    for base, img_fname in img_map.items():
        if base in mask_map:
            pairs.append((
                os.path.join(images_dir, img_fname),
                os.path.join(masks_dir, mask_map[base]),
            ))
    return pairs


def mask_to_yolo_boxes(mask_path, img_w, img_h):
    """Return list of (x_center, y_center, w, h), all normalized 0-1."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    if mask.shape[:2] != (img_h, img_w):
        mask = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)
    binary = (mask > 127).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < MIN_CONTOUR_AREA:
            continue
        x, y, w, h = cv2.boundingRect(c)
        xc = (x + w / 2) / img_w
        yc = (y + h / 2) / img_h
        boxes.append((xc, yc, w / img_w, h / img_h))
    return boxes


def main():
    random.seed(SEED)

    for split in ("train", "val"):
        os.makedirs(os.path.join(OUTPUT_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, "labels", split), exist_ok=True)

    total_written = 0
    total_empty_skipped = 0
    per_source_counts = {}

    for source in SOURCES:
        pairs = find_pairs(source["images_dir"], source["masks_dir"])
        print(f"[{source['name']}] found {len(pairs)} image/mask pairs at:\n"
              f"    images: {source['images_dir']}\n"
              f"    masks:  {source['masks_dir']}")

        if not pairs:
            print(f"  !! No pairs found for {source['name']} -- check the paths above still exist.")
            per_source_counts[source["name"]] = 0
            continue

        random.shuffle(pairs)
        n_val = max(1, int(len(pairs) * VAL_FRACTION))
        val_pairs = pairs[:n_val]
        train_pairs = pairs[n_val:]

        written_this_source = 0
        for split, split_pairs in (("train", train_pairs), ("val", val_pairs)):
            for img_path, mask_path in split_pairs:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                h, w = img.shape[:2]

                boxes = mask_to_yolo_boxes(mask_path, w, h)
                if not boxes:
                    total_empty_skipped += 1
                    continue  # skip frames with no detectable polyp blob

                # unique filename across sources to avoid collisions
                out_stem = f"{source['name']}_{os.path.splitext(os.path.basename(img_path))[0]}"
                out_img_path = os.path.join(OUTPUT_DIR, "images", split, out_stem + ".png")
                out_lbl_path = os.path.join(OUTPUT_DIR, "labels", split, out_stem + ".txt")

                cv2.imwrite(out_img_path, img)
                with open(out_lbl_path, "w") as f:
                    for xc, yc, bw, bh in boxes:
                        f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

                written_this_source += 1
                total_written += 1

        per_source_counts[source["name"]] = written_this_source
        print(f"  -> wrote {written_this_source} labeled images ({len(train_pairs)} train / {len(val_pairs)} val target split)")

    # dataset.yaml
    yaml_path = os.path.join(OUTPUT_DIR, "dataset.yaml")
    with open(yaml_path, "w") as f:
        yaml.dump({
            "path": OUTPUT_DIR,
            "train": "images/train",
            "val": "images/val",
            "names": {0: "polyp"},
        }, f)

    print("\n" + "=" * 60)
    print("BUILD SUMMARY")
    print("=" * 60)
    for name, count in per_source_counts.items():
        print(f"  {name:25s} {count} labeled images")
    print(f"  Total written: {total_written}")
    print(f"  Skipped (empty mask / no contour): {total_empty_skipped}")
    print(f"  Output dataset root: {OUTPUT_DIR}")
    print(f"  dataset.yaml: {yaml_path}")
    print("\nPoint kaggle_yolo_retrain.py's dataset_path at:")
    print(f"  {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

[cvc-clinicdb] found 612 image/mask pairs at:
    images: /kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Original
    masks:  /kaggle/input/datasets/balraj98/cvcclinicdb/PNG/Ground Truth
  -> wrote 612 labeled images (521 train / 91 val target split)
[cvc-300] found 60 image/mask pairs at:
    images: /kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset/CVC-300/images
    masks:  /kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset/CVC-300/masks
  -> wrote 60 labeled images (51 train / 9 val target split)
[etis-laribpolypdb] found 196 image/mask pairs at:
    images: /kaggle/input/datasets/nguyenvoquocduong/etis-laribpolypdb/images
    masks:  /kaggle/input/datasets/nguyenvoquocduong/etis-laribpolypdb/masks
  -> wrote 196 labeled images (167 train / 29 val target split)

BUILD SUMMARY
  cvc-clinicdb              612 labeled images
  cvc-300                   60 labeled images
  etis-laribpolypdb         196 labeled images
  Total written: 868
  Skipped 

In [2]:
import os
import yaml
import multiprocessing
import subprocess
import sys

# 1. Install dependencies if missing BEFORE importing
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics"], check=True)
    import ultralytics

from ultralytics import YOLO
import torch


def main():
    # ==========================================
    # DATASET COMPOSITION WARNING
    # Do NOT train on a dataset containing only ETIS-Larib images!
    # ACTION: Make sure `chakramodel-yolo-combo-dataset` contains a COMBINED dataset.
    # ==========================================

    # Kaggle paths
    # Built by build_yolo_combo_dataset.py -- run that script first this session
    dataset_path = "/kaggle/working/chakramodel_yolo_combo_dataset"
    working_dir = "/kaggle/working"

    if not os.path.exists(dataset_path):
        print(f"ERROR: Dataset not found at {dataset_path}")
        print("Please upload your COMBINED YOLO-formatted dataset to Kaggle and update the 'dataset_path' variable.")
        return

    data_yaml_path = os.path.join(working_dir, "dataset.yaml")

    yaml_content = {
        "path": dataset_path,
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "polyp"
        }
    }

    with open(data_yaml_path, 'w') as f:
        yaml.dump(yaml_content, f)

    weight_dir_options = [
        "/kaggle/input/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
        "/kaggle/input/datasets/gokulraj324/chakramodel-weights",  # confirmed location this session
        "/kaggle/working",
        "."
    ]

    yolo_weight = None
    for d in weight_dir_options:
        if os.path.exists(os.path.join(d, "best.pt")):
            yolo_weight = os.path.join(d, "best.pt")
            break

    if not yolo_weight:
        print("ERROR: Existing YOLO weight 'best.pt' not found in typical Kaggle paths!")
        return

    print(f"Loading existing fine-tuned weights from {yolo_weight}")
    model = YOLO(yolo_weight)

    print("Starting YOLOv8x Retraining for ETIS-Larib Domain Adaptation...")

    # Gracefully handle CPU-only instances
    device = 0 if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: No GPU found. Training on CPU will be extremely slow.")

    # ---------------------------------------------------------
    # PATH B: AGGRESSIVE AUGMENTATION CONFIGURATION
    # ---------------------------------------------------------
    results = model.train(
        data=data_yaml_path,
        epochs=30,
        imgsz=384,
        batch=16,
        workers=4,
        device=device,
        amp=True,
        project=working_dir,
        name="polyp_yolov8x_etis_fix",
        patience=20,
        save=True,
        exist_ok=True,

        # Explicit seed for reproducible benchmarking
        seed=42,

        # AGGRESSIVE AUGMENTATIONS FOR NBI/LCI/BLI (Path B Fix)
        hsv_h=0.1,
        hsv_s=0.9,
        hsv_v=0.9,
        degrees=45.0,
        translate=0.2,
        scale=0.5,
        shear=0.1,
        perspective=0.001,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2
    )

    print("Training finished! Models saved to /kaggle/working/polyp_yolov8x_etis_fix/weights/")


if __name__ == '__main__':
    multiprocessing.freeze_support()
    main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Loading existing fine-tuned weights from /kaggle/input/datasets/gokulraj324/chakramodel-weights/best.pt
Starting YOLOv8x Retraining for ETIS-Larib Domain Adaptation...
Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      1.04G      1.758      1.647      1.635          6        384: 100% ━━━━━━━━━━━━ 47/47 8.6it/s 5.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 7.8it/s 0.6s
                   all        129        141      0.856      0.766      0.862      0.322

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      1.04G      1.731      1.634      1.645          5        384: 100% ━━━━━━━━━━━━ 47/47 8.2it/s 5.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 9.2it/s 0.5s
                   all        129        141       0.86      0.743      0.833       0.28

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      1.04G      1.717      1.652      1.608         13        384: 100% ━━━━━━━━━━━━ 47/47 8.4it/s 5.6s
                 Class     Images  Instances      Box(P          R

In [3]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
import subprocess
import sys

# 1. Install dependencies if missing BEFORE importing them
try:
    import ultralytics
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "timm"], check=True)
    import ultralytics
    import timm

from ultralytics import YOLO
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# PATH A: HARDENED INFERENCE CONFIGURATION
# ==========================================
# Use the exact same dataset path you used for the baseline 0.5401 score
dataset_path = "/kaggle/input/datasets/balraj98/cvcclinicdb/PNG"  # confirmed location this session

# We will sweep across multiple thresholds to find the optimum!
CONFIDENCE_THRESHOLDS = [0.25, 0.40, 0.50, 0.65, 0.80]
# ==========================================

print(f"Scanning dataset at: {dataset_path}")

# Check for weights in typical kaggle paths
weight_dir_options = [
    "/kaggle/input/chakramodel-weights",
    "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
    "/kaggle/input/datasets/gokulraj324/chakramodel-weights",  # confirmed location this session
    "/kaggle/working",
    "."
]

yolo_weight = None
vit_weight = None

for d in weight_dir_options:
    if os.path.exists(os.path.join(d, "best.pt")):
        yolo_weight = os.path.join(d, "best.pt")
    if os.path.exists(os.path.join(d, "chakra_transformer_best.pth")):
        vit_weight = os.path.join(d, "chakra_transformer_best.pth")

if not yolo_weight or not vit_weight:
    print("ERROR: Weights not found in Kaggle! Please upload best.pt and chakra_transformer_best.pth")
    sys.exit(1)

print(f"Found YOLO weight at {yolo_weight}")
print(f"Found ViT weight at {vit_weight}")

# 3. Model definitions
class ChakraTransformerSegmenter(nn.Module):
    def __init__(self, backbone_name='vit_large_patch16_384', pretrained=False, num_classes=1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=False)
        self.embed_dim = self.backbone.embed_dim
        self.decode_head = nn.Sequential(
            nn.ConvTranspose2d(self.embed_dim, 256, kernel_size=4, stride=4),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=4),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1),
        )
        self.dropout1 = nn.Dropout2d(p=0.5)
        self.dropout2 = nn.Dropout2d(p=0.5)

    def forward(self, x):
        b, c, h, w = x.shape
        features = self.backbone.forward_features(x)
        if features.dim() == 3:
            grid_h, grid_w = h // 16, w // 16
            expected = grid_h * grid_w
            if features.shape[1] == expected + 1:
                features = features[:, 1:, :]
            elif features.shape[1] > expected:
                features = features[:, :expected, :]
            features = features.transpose(1, 2).contiguous().view(b, self.embed_dim, grid_h, grid_w)

        x_dec = features
        for i, layer in enumerate(self.decode_head):
            x_dec = layer(x_dec)
            if i == 2: x_dec = self.dropout1(x_dec)
            elif i == 5: x_dec = self.dropout2(x_dec)

        logits = x_dec
        if logits.shape[2:] != (h, w):
            logits = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
        return logits


def letterbox(img, new_shape=(384, 384), color=(0, 0, 0)):
    shape = img.shape[:2]
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
    dw, dh = dw / 2, dh / 2
    if shape[::-1] != new_unpad:
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return img, (dw, dh, r)


def calculate_dice(pred, target):
    smooth = 1e-5
    pred_f = pred.flatten()
    target_f = target.flatten()
    intersection = np.sum(pred_f * target_f)
    return (2. * intersection + smooth) / (np.sum(pred_f) + np.sum(target_f) + smooth)


# ---------------------------------------------------------
# Fix: generalized mask<->image matching.
# Instead of hardcoding a couple of suffixes, strip any known
# mask-indicator token from EITHER end of the mask filename and
# compare the normalized base against image bases. Covers
# "_mask", "_gt", "_manual", "_label", "_seg", "_segmentation",
# "mask_", "gt_", "label_" and combinations, without risking the
# old numeric-substring collisions (e.g. "1" matching "10").
# ---------------------------------------------------------
MASK_TOKENS = ["mask", "ground", "gt", "label", "seg", "segmentation", "manual"]


def normalize_mask_base(filename_base):
    base = filename_base
    changed = True
    while changed:
        changed = False
        low = base.lower()
        for tok in MASK_TOKENS:
            for pattern in (f"_{tok}", f"-{tok}", tok):
                if low.startswith(pattern):
                    base = base[len(pattern):]
                    low = base.lower()
                    changed = True
                elif low.endswith(pattern):
                    base = base[: -len(pattern)]
                    low = base.lower()
                    changed = True
        base = base.strip("_- ")
    return base


print("Loading YOLO stage 1...")
yolo_model = YOLO(yolo_weight)

print("Loading ChakraTransformer stage 2...")
segmenter = ChakraTransformerSegmenter('vit_large_patch16_384', pretrained=False, num_classes=1)
state_dict = torch.load(vit_weight, map_location='cpu', weights_only=True)
if "model" in state_dict: state_dict = state_dict["model"]
new_state_dict = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in state_dict.items()}

load_result = segmenter.load_state_dict(new_state_dict, strict=False)
print("ViT Weights Load Result:")
if load_result.missing_keys:
    print(f"  Missing Keys: {len(load_result.missing_keys)}")
if load_result.unexpected_keys:
    print(f"  Unexpected Keys: {len(load_result.unexpected_keys)}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
segmenter.to(DEVICE)
segmenter.eval()

# Precompute normalization tensors once
norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

# Generalized dataset loading
# NOTE: mask/image separation is decided from the FULL PATH, not just the
# filename -- several real datasets (e.g. CVC-ClinicDB) use plain numeric
# filenames like "1.png" and instead put the distinction in the folder
# name ("Ground Truth" vs "Original"). Checking only the basename misses
# these and silently produces zero matched pairs.
all_files = []
for root, _, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
            all_files.append(os.path.join(root, f))

mask_files = []
img_files = []
for f in all_files:
    low = f.lower()  # full path, so folder names like "Ground Truth/" are caught
    if any(tok in low for tok in ("mask", "ground", "gt", "label")):
        mask_files.append(f)
    else:
        img_files.append(f)

# Build lookup: normalized mask base -> mask path
mask_lookup = {}
unresolved_masks = []
for m in mask_files:
    m_base = os.path.splitext(os.path.basename(m))[0]
    norm = normalize_mask_base(m_base)
    if norm in mask_lookup:
        unresolved_masks.append(m)  # collision, skip to avoid mispairing
    else:
        mask_lookup[norm] = m

matched_pairs = []
unmatched_images = []
for img_path in img_files:
    base = os.path.splitext(os.path.basename(img_path))[0]
    matched_mask = mask_lookup.get(base)
    if matched_mask:
        matched_pairs.append((img_path, matched_mask))
    else:
        unmatched_images.append(img_path)

print(f"Successfully matched {len(matched_pairs)} image-mask pairs.")
if unmatched_images:
    print(f"WARNING: {len(unmatched_images)} images had no mask match "
          f"(e.g. {[os.path.basename(p) for p in unmatched_images[:5]]})")
if unresolved_masks:
    print(f"WARNING: {len(unresolved_masks)} masks had ambiguous/colliding normalized names "
          f"and were skipped (e.g. {[os.path.basename(p) for p in unresolved_masks[:5]]})")

# Cache inference once at the minimum threshold, sweep thresholds against cached boxes.
# Safe because Ultralytics NMS only ever suppresses a box using a higher-confidence
# box already in the candidate pool -- adding lower-confidence boxes to that pool
# never changes which higher-confidence boxes survive. So filtering the cached
# min-threshold boxes by a higher threshold reproduces exactly what a direct
# higher-threshold run would keep.
results_by_threshold = {t: {"dices": [], "zero_skips": 0} for t in CONFIDENCE_THRESHOLDS}
MIN_THRESH = min(CONFIDENCE_THRESHOLDS)

print(f"\n--- Running cached pipeline inference (YOLO base conf={MIN_THRESH}) ---")

with torch.no_grad():
    for img_path, mask_path in tqdm(matched_pairs):
        img_bgr = cv2.imread(img_path)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img_bgr is None or gt_mask is None:
            continue

        orig_h, orig_w = gt_mask.shape[:2]

        if np.max(gt_mask) == 1:
            gt_mask = gt_mask * 255
        gt_bin = (gt_mask > 127).astype(np.uint8)

        # Negative (no-polyp) frames are kept in the eval; calculate_dice's
        # smoothing term correctly scores an all-zero pred against an
        # all-zero gt as dice ~= 1.0.

        img_resized = cv2.resize(img_bgr, (384, 384))
        yolo_results = yolo_model(img_resized, conf=MIN_THRESH, imgsz=384, verbose=False, device=DEVICE)[0]

        cached_masks = []
        for box_data in yolo_results.boxes:
            conf = float(box_data.conf[0].cpu())
            box = box_data.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)

            scale_x = orig_w / 384.0
            scale_y = orig_h / 384.0

            gx, gy, gx2, gy2 = int(x1 * scale_x), int(y1 * scale_y), int(x2 * scale_x), int(y2 * scale_y)
            gw, gh = gx2 - gx, gy2 - gy

            px, py = int(0.25 * gw), int(0.25 * gh)
            px1, py1 = max(0, gx - px), max(0, gy - py)
            px2, py2 = min(orig_w, gx + gw + px), min(orig_h, gy + gh + py)

            roi_crop = img_bgr[py1:py2, px1:px2]
            if roi_crop.size == 0:
                continue

            roi_crop_resized, (dw, dh, r) = letterbox(roi_crop, new_shape=(384, 384))
            roi_crop_rgb = cv2.cvtColor(roi_crop_resized, cv2.COLOR_BGR2RGB)

            t_roi = torch.from_numpy(roi_crop_rgb).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE) / 255.0
            t_roi = (t_roi - norm_mean) / norm_std

            out = segmenter(t_roi)
            if isinstance(out, tuple):
                out = out[0]
            prob = torch.sigmoid(out).float().squeeze().cpu().numpy()
            seg_roi_mask = (prob > 0.5).astype(np.uint8)

            top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
            left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
            unpad_h = 384 - top - bottom
            unpad_w = 384 - left - right

            seg_roi_mask_unpad = seg_roi_mask[top:top + unpad_h, left:left + unpad_w]
            seg_roi_mask_resized = cv2.resize(seg_roi_mask_unpad, (px2 - px1, py2 - py1), interpolation=cv2.INTER_NEAREST)

            cached_masks.append({
                "conf": conf,
                "mask": seg_roi_mask_resized,
                "coords": (py1, py2, px1, px2)
            })

        for thresh in CONFIDENCE_THRESHOLDS:
            full_pred = np.zeros((orig_h, orig_w), dtype=np.uint8)
            passed = 0

            for c_mask in cached_masks:
                if c_mask["conf"] >= thresh:
                    py1, py2, px1, px2 = c_mask["coords"]
                    full_pred[py1:py2, px1:px2] = np.maximum(
                        full_pred[py1:py2, px1:px2],
                        c_mask["mask"]
                    )
                    passed += 1

            if passed == 0:
                results_by_threshold[thresh]["zero_skips"] += 1

            dice = calculate_dice(full_pred, gt_bin)
            results_by_threshold[thresh]["dices"].append(dice)

print("\n==============================================")
print("--- Confidence Threshold Sweep Results ---")
print("==============================================")
for thresh in CONFIDENCE_THRESHOLDS:
    avg_dice = np.mean(results_by_threshold[thresh]["dices"]) if results_by_threshold[thresh]["dices"] else 0.0
    zero_skips = results_by_threshold[thresh]["zero_skips"]
    print(f"Threshold: {thresh:.2f} | Dice: {avg_dice:.4f} | Zero Detections: {zero_skips}")
print("==============================================")

Scanning dataset at: /kaggle/input/datasets/balraj98/cvcclinicdb/PNG
Found YOLO weight at /kaggle/input/datasets/gokulraj324/chakramodel-weights/best.pt
Found ViT weight at /kaggle/input/datasets/gokulraj324/chakramodel-weights/chakra_transformer_best.pth
Loading YOLO stage 1...
Loading ChakraTransformer stage 2...
ViT Weights Load Result:
Successfully matched 612 image-mask pairs.

--- Running cached pipeline inference (YOLO base conf=0.25) ---


100%|██████████| 612/612 [01:33<00:00,  6.56it/s]


--- Confidence Threshold Sweep Results ---
Threshold: 0.25 | Dice: 0.9109 | Zero Detections: 2
Threshold: 0.40 | Dice: 0.9075 | Zero Detections: 4
Threshold: 0.50 | Dice: 0.8995 | Zero Detections: 10
Threshold: 0.65 | Dice: 0.8801 | Zero Detections: 24
Threshold: 0.80 | Dice: 0.8157 | Zero Detections: 69


In [4]:
import os
import csv
import cv2
import numpy as np
from collections import defaultdict

# ==========================================================
# DEEP KAGGLE DATASET DIAGNOSTIC (A-to-Z)
# Run this FIRST, right after attaching your datasets.
# Goes beyond folder listing: opens sample images/masks,
# reports real dimensions/channels/value ranges, classifies
# each dataset as SEGMENTATION vs YOLO-DETECTION vs UNKNOWN,
# checks weight checkpoints, and previews any CSVs.
# ==========================================================

ROOT = "/kaggle/input"
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
WEIGHT_EXTS = (".pt", ".pth", ".onnx", ".engine")
MASK_TOKENS = ("mask", "ground", "gt", "label", "seg", "segmentation", "manual")
SAMPLE_N = 5
MAX_TREE_DEPTH = 6


def human_size(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


def is_image(fname):
    return fname.lower().endswith(IMG_EXTS)


def is_weight(fname):
    return fname.lower().endswith(WEIGHT_EXTS)


def looks_like_mask_name(name):
    low = name.lower()
    return any(tok in low for tok in MASK_TOKENS)


def inspect_image(path):
    """Return (h, w, channels, dtype, min, max) or None on failure."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 2:
        h, w = img.shape
        c = 1
    else:
        h, w, c = img.shape
    return (h, w, c, str(img.dtype), int(img.min()), int(img.max()))


def inspect_weight_file(path):
    """Best-effort peek into a torch checkpoint without crashing the script."""
    try:
        import torch
        obj = torch.load(path, map_location="cpu", weights_only=True)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            if "model" in obj and isinstance(obj["model"], dict):
                inner_keys = list(obj["model"].keys())
                return f"dict with top-level keys {keys[:5]}{'...' if len(keys) > 5 else ''}; " \
                       f"'model' sub-dict has {len(inner_keys)} tensor keys, e.g. {inner_keys[:3]}"
            return f"state_dict with {len(keys)} keys, e.g. {keys[:3]}"
        return f"loaded object of type {type(obj)}"
    except Exception as e:
        return f"could not introspect ({type(e).__name__}: {e})"


def preview_csv(path, n=SAMPLE_N):
    try:
        with open(path, newline="", encoding="utf-8", errors="replace") as f:
            reader = csv.reader(f)
            rows = []
            for i, row in enumerate(reader):
                if i >= n:
                    break
                rows.append(row)
        return rows
    except Exception as e:
        return [[f"<could not read: {e}>"]]


def classify_directory_set(files_in_tree):
    """
    Given all file paths under one top-level dataset, decide what kind
    of dataset it looks like: YOLO-detection, segmentation (img+mask),
    plain image collection, or unknown/mixed.
    """
    has_yolo_split = any(
        ("images" + os.sep + "train") in f or ("images" + os.sep + "val") in f
        for f in files_in_tree
    )
    has_label_txt = any(f.lower().endswith(".txt") and "label" in f.lower() for f in files_in_tree)
    has_masks = any(looks_like_mask_name(os.path.basename(f)) for f in files_in_tree if is_image(f))
    has_images = any(is_image(f) and not looks_like_mask_name(os.path.basename(f)) for f in files_in_tree)

    if has_yolo_split and has_label_txt:
        return "YOLO-DETECTION (images/train+val with .txt labels)"
    if has_yolo_split and not has_label_txt:
        return "YOLO-STYLE FOLDERS BUT NO LABEL FILES FOUND (incomplete for training)"
    if has_masks and has_images:
        return "SEGMENTATION (image + mask pairs, no bbox labels)"
    if has_images:
        return "PLAIN IMAGE COLLECTION (no masks, no labels)"
    return "UNKNOWN / NON-IMAGE DATA"


if not os.path.exists(ROOT):
    print(f"!! {ROOT} does not exist. Are you running this on Kaggle?")
else:
    top_entries = sorted(os.listdir(ROOT))
    print("=" * 72)
    print(f"TOP-LEVEL CONTENTS OF {ROOT}  ({len(top_entries)} entries)")
    print("=" * 72)
    for e in top_entries:
        full = os.path.join(ROOT, e)
        print(f"  [{'DIR ' if os.path.isdir(full) else 'FILE'}] {e}")

    # Walk everything once; group by the deepest "dataset-looking" folder
    # (Kaggle convention: /kaggle/input/datasets/<user>/<dataset-slug>/...
    #  or /kaggle/input/<dataset-slug>/...)
    dataset_groups = defaultdict(list)

    for dirpath, dirnames, filenames in os.walk(ROOT):
        for f in filenames:
            full_path = os.path.join(dirpath, f)
            rel = os.path.relpath(full_path, ROOT)
            parts = rel.split(os.sep)
            # group key = up to 3 path segments deep (covers both
            # /input/<slug>/... and /input/datasets/<user>/<slug>/...)
            if parts[0] == "datasets" and len(parts) > 2:
                group_key = os.sep.join(parts[:3])
            else:
                group_key = parts[0]
            dataset_groups[group_key].append(full_path)

    print("\n" + "=" * 72)
    print("PER-DATASET DEEP INSPECTION")
    print("=" * 72)

    for group_key in sorted(dataset_groups.keys()):
        files = dataset_groups[group_key]
        print(f"\n--- {group_key} ---")

        total_size = sum(os.path.getsize(f) for f in files if os.path.exists(f))
        print(f"  Files: {len(files)}  |  Size: {human_size(total_size)}")

        ext_counts = defaultdict(int)
        for f in files:
            ext = os.path.splitext(f)[1].lower() or "(no ext)"
            ext_counts[ext] += 1
        print(f"  Extensions: {dict(sorted(ext_counts.items(), key=lambda x: -x[1]))}")

        classification = classify_directory_set(files)
        print(f"  CLASSIFICATION: {classification}")

        # weight files: full introspection since these gate the whole pipeline
        weight_files = [f for f in files if is_weight(f)]
        for wf in weight_files:
            print(f"  WEIGHT: {wf}  ({human_size(os.path.getsize(wf))})")
            print(f"          -> {inspect_weight_file(wf)}")

        # image / mask content inspection (sample a few real files)
        image_files = [f for f in files if is_image(f) and not looks_like_mask_name(os.path.basename(f))]
        mask_files = [f for f in files if is_image(f) and looks_like_mask_name(os.path.basename(f))]

        if image_files:
            print(f"  Sample image inspection ({min(SAMPLE_N, len(image_files))} of {len(image_files)}):")
            for f in image_files[:SAMPLE_N]:
                info = inspect_image(f)
                if info:
                    h, w, c, dtype, mn, mx = info
                    print(f"    {os.path.basename(f):30s} {w}x{h}  ch={c}  dtype={dtype}  range=[{mn},{mx}]")
                else:
                    print(f"    {os.path.basename(f):30s} <failed to open>")

        if mask_files:
            print(f"  Sample mask inspection ({min(SAMPLE_N, len(mask_files))} of {len(mask_files)}):")
            for f in mask_files[:SAMPLE_N]:
                info = inspect_image(f)
                if info:
                    h, w, c, dtype, mn, mx = info
                    binary_hint = "binary-like (0/255 or 0/1)" if mx <= 255 and mn == 0 and (mx in (1, 255)) else "check manually"
                    print(f"    {os.path.basename(f):30s} {w}x{h}  ch={c}  dtype={dtype}  range=[{mn},{mx}]  {binary_hint}")
                else:
                    print(f"    {os.path.basename(f):30s} <failed to open>")

        # image/mask count parity check
        if image_files and mask_files:
            if len(image_files) != len(mask_files):
                print(f"  !! COUNT MISMATCH: {len(image_files)} images vs {len(mask_files)} masks -- some are likely unpaired")
            else:
                print(f"  Image/mask counts match: {len(image_files)} each")

        # CSV preview
        csv_files = [f for f in files if f.lower().endswith(".csv")]
        for cf in csv_files:
            print(f"  CSV: {cf}")
            for row in preview_csv(cf):
                print(f"    {row}")

    print("\n" + "=" * 72)
    print("PATHS COMMONLY HARDCODED IN CHAKRAMODEL SCRIPTS -- verify these")
    print("=" * 72)
    candidate_paths = [
        "/kaggle/input/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-weights",
        "/kaggle/input/datasets/gokulraj324/chakramodel-weights",
        "/kaggle/input/datasets/gokulrocky/chakramodel-yolo-combo-dataset",
        "/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/cvc-clinicdb",
        "/kaggle/working",
    ]
    for p in candidate_paths:
        exists = os.path.exists(p)
        print(f"  [{'OK  ' if exists else 'MISSING'}] {p}")

    print("\nDone. Use the CLASSIFICATION lines above to know which scripts can run as-is")
    print("(SEGMENTATION datasets need bbox generation before YOLO training; only a")
    print("YOLO-DETECTION dataset with images/train+val AND label .txt files is ready to train on).")

TOP-LEVEL CONTENTS OF /kaggle/input  (1 entries)
  [DIR ] datasets

PER-DATASET DEEP INSPECTION

--- datasets/balraj98/cvcclinicdb ---
  Files: 2450  |  Size: 309.5MB
  Extensions: {'.tif': 1224, '.png': 1224, '.csv': 2}
  CLASSIFICATION: PLAIN IMAGE COLLECTION (no masks, no labels)
  Sample image inspection (5 of 2448):
    274.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    315.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    123.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    130.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
    508.tif                        384x288  ch=1  dtype=uint8  range=[0,255]
  CSV: /kaggle/input/datasets/balraj98/cvcclinicdb/class_dict.csv
    ['class_names', 'r', 'g', 'b']
    ['background', '0', '0', '0']
    ['polyp', '255', '255', '255']
  CSV: /kaggle/input/datasets/balraj98/cvcclinicdb/metadata.csv
    ['frame_id', 'sequence_id', 'tif_image_path', 'tif_mas